# LEVELS Benchmark: DSI Data Explorer Agent

This notebook benchmarks the DSI Data Explorer agent across three task-complexity levels and five data-management categories.

## Complexity Levels

| Level | Name | Description | Scoring |
|-------|------|-------------|---------|
| **1** | Atomic | Single DSI/API call from natural language | Binary pass/fail: correct tool + task completion |
| **2** | Workflow | Multi-step pipelines (query → filter → summarize, ingest → validate, etc.) | Output correctness + method quality (LLM judge) |
| **3** | Discovery | Hypothesis-level: state a relationship as structured hypothesis (DiscoveryBench-style) | Faceted LLM judge + expert ground truth |

## Data Management Categories

1. Search — external/domain knowledge and literature
2. Data Discovery — locating datasets, files, or records in DSI repositories
3. Data Preparation — cleaning, validating, standardizing scientific data
4. Metadata Generation — structured documentation and provenance
5. Analysis & Summarization — interpreting, aggregating, synthesizing data

## Setup

In [2]:
# Install dependencies into the active Jupyter kernel (run once)
%pip install -q -r requirements.txt
%pip install -q -e ../..

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Paste your portal API Key here
import os
os.environ.setdefault("OPENAI_API_KEY", "sk-gxw7Gyjb6tCpzcEXQPg14Q")
os.environ["OPENAI_API_BASE"] = "https://aiportal-api.aws.lanl.gov"

In [12]:
import json
import re
import sys
import time
from dataclasses import dataclass, field, asdict
from datetime import datetime
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional

import pandas as pd
from IPython.display import Markdown, display
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI

def find_ai_search_dir():
    """Locate tools/ai_search even if cwd was changed by a prior agent run."""
    candidates = [Path.cwd(), *Path.cwd().parents]
    for directory in candidates:
        if (directory / "dsi_explorer_agent.py").exists():
            return directory.resolve(), directory.parent.parent.resolve()
        repo_candidate = directory / "tools" / "ai_search" / "dsi_explorer_agent.py"
        if repo_candidate.exists():
            return repo_candidate.parent.resolve(), directory.resolve()
    raise RuntimeError(
        "Could not find tools/ai_search/dsi_explorer_agent.py. "
        "Open this notebook from the dsi repo, or run: cd tools/ai_search"
    )

AI_SEARCH_DIR, REPO_ROOT = find_ai_search_dir()

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(AI_SEARCH_DIR)

from dsi_explorer_agent import DSIExplorer, extract_message_texts

print(f"Working directory: {AI_SEARCH_DIR}")
print(f"Repo root: {REPO_ROOT}")

Working directory: /Users/hhn/lanl/dsi/tools/ai_search
Repo root: /Users/hhn/lanl/dsi


In [ ]:
# --- Configuration ---
AI_PORTAL_BASE_URL = os.environ.get("OPENAI_API_BASE", "https://aiportal-api.aws.lanl.gov")

POSSIBLE_MODELS = [
    "aws-gov.gpt-5.4",
    "anthropic.claude-sonnet-4-5-20250929-v1:0",
    "amazon.nova-pro-v1:0",
    "meta.llama3-70b-instruct-v1:0",
    "meta.llama3-8b-instruct-v1:0",
    "mistral7b",
    "llama3bv32instr",
    "phi4",
    "gpt-oss-120b",
    "e5mistral7b",
    "NVIDIA-Nemotron-3-Super-120B-A12B-FP8",
    "gemma-4-31B-it",
]

LLM_MODEL = "aws-gov.gpt-5.4"   # change to any model in AVAILABLE_MODELS
JUDGE_MODEL = LLM_MODEL         # LLM judge for Level 2/3
RUN_MODE = "smoke"              # "smoke" = 1 task per category/level; "full" = all tasks
USE_MCP = False                 # set True if mcp_wrapper_server.py is running on :8000
MCP_URL = "http://127.0.0.1:8000/mcp"

MASTER_DB = "data/oceans_11/ocean_11_datasets.db"
NIF_DB = "data/oceans_11/nif.db"
POISSON_DB = "data/oceans_11/poisson.db"

RESULTS_DIR = AI_SEARCH_DIR / "levels_benchmark_results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

UNCERTAINTY_MARKERS = re.compile(
    r"\b(uncertain|unsure|don't know|do not know|cannot determine|not sure|"
    r"may need|might need|human|expert|clarif|ask you|cannot guarantee|"
    r"limited information|insufficient|unable to confirm)\b",
    re.IGNORECASE,
)

def make_llm(model_name: str, temperature: float = 0.1) -> ChatOpenAI:
    return ChatOpenAI(
        model=model_name,
        api_key=os.environ["OPENAI_API_KEY"],
        base_url=AI_PORTAL_BASE_URL,
        temperature=temperature,
    )

llm_model = make_llm(LLM_MODEL)
judge_llm = make_llm(JUDGE_MODEL, temperature=0.0)
print(f"Using model: {LLM_MODEL} @ {AI_PORTAL_BASE_URL}")

Using model: aws-gov.gpt-5.4 @ https://aiportal-api.aws.lanl.gov


In [14]:
def make_explorer() -> DSIExplorer:
    """Create a fresh DSIExplorer instance (new workspace + thread)."""
    os.chdir(AI_SEARCH_DIR)  # avoid nested workspace folders on re-runs
    kwargs = dict(llm=llm_model, db_index_name=MASTER_DB, output_mode="jupyter")
    if USE_MCP:
        kwargs.update(use_mcp=True, mcp_transport="streamable-http", mcp_url=MCP_URL)
    return DSIExplorer(**kwargs)


ai = make_explorer()
print(f"Agent workspace: {ai.wrsk_path}")

Creating workspace at: /Users/hhn/lanl/dsi/tools/ai_search/dsi_explorer_agent__2026_06_30__11_25
Agent workspace: /Users/hhn/lanl/dsi/tools/ai_search/dsi_explorer_agent__2026_06_30__11_25


## Benchmark Harness

In [15]:
@dataclass
class BenchmarkTask:
    task_id: str
    level: int
    category: str
    query: str
    description: str = ""
    # Level 1 checks
    expected_tools: List[str] = field(default_factory=list)
    response_keywords: List[str] = field(default_factory=list)
    max_tool_calls: Optional[int] = None
    setup_query: Optional[str] = None
    # Level 2/3 rubric
    success_criteria: str = ""
    ground_truth_hypothesis: Optional[Dict[str, str]] = None
    requires_network: bool = False
    skip: bool = False


@dataclass
class TaskResult:
    task_id: str
    level: int
    category: str
    query: str
    response: str
    tool_calls: List[Dict[str, Any]]
    elapsed_sec: float
    tokens: int
    passed: Optional[bool] = None
    score: Optional[float] = None
    judge_scores: Optional[Dict[str, Any]] = None
    notes: str = ""


def extract_tool_calls(result: dict) -> List[Dict[str, Any]]:
    calls = []
    for msg in result.get("messages", []):
        if isinstance(msg, AIMessage) and msg.tool_calls:
            for tc in msg.tool_calls:
                calls.append({"name": tc.get("name"), "args": tc.get("args", {})})
    return calls


def invoke_agent(explorer: DSIExplorer, query: str) -> dict:
    start = time.time()
    msg = explorer.craft_message(query)
    if explorer.use_mcp:
        from dsi_explorer_agent import run_async
        result = run_async(explorer.app.ainvoke(
            msg, config={"configurable": {"thread_id": explorer.thread_id}}
        ))
    else:
        result = explorer.app.invoke(
            msg, config={"configurable": {"thread_id": explorer.thread_id}}
        )
    result["_elapsed"] = time.time() - start
    return result


def run_task(explorer: DSIExplorer, task: BenchmarkTask, verbose: bool = True) -> TaskResult:
    if task.skip:
        return TaskResult(task.task_id, task.level, task.category, task.query,
                          "", [], 0.0, 0, passed=None, notes="skipped")

    if task.setup_query:
        invoke_agent(explorer, task.setup_query)

    if verbose:
        display(Markdown(f"### `{task.task_id}` — Level {task.level} / {task.category}\n\n**Query:** {task.query}"))

    result = invoke_agent(explorer, task.query)
    response = (result.get("response") or "").strip()
    tool_calls = extract_tool_calls(result)
    tokens = result.get("metadata", {}).get("token_usage", {}).get("total_tokens", 0)

    if verbose:
        display(Markdown(response))
        print(f"Tools called: {[t['name'] for t in tool_calls]} | {result['_elapsed']:.1f}s | {tokens} tokens")

    return TaskResult(
        task_id=task.task_id,
        level=task.level,
        category=task.category,
        query=task.query,
        response=response,
        tool_calls=tool_calls,
        elapsed_sec=result["_elapsed"],
        tokens=tokens,
    )


def score_level1(task: BenchmarkTask, result: TaskResult) -> TaskResult:
    """Atomic scoring: correct tool selection + basic completion signals."""
    tool_names = [t["name"] for t in result.tool_calls]
    notes = []

    tool_ok = True
    if task.expected_tools:
        tool_ok = any(t in tool_names for t in task.expected_tools)
        if not tool_ok:
            notes.append(f"Expected one of {task.expected_tools}, got {tool_names}")

    keyword_ok = True
    if task.response_keywords:
        response_lower = result.response.lower()
        keyword_ok = any(kw.lower() in response_lower for kw in task.response_keywords)
        if not keyword_ok:
            notes.append(f"Response missing keywords: {task.response_keywords}")

    count_ok = True
    if task.max_tool_calls is not None and len(result.tool_calls) > task.max_tool_calls:
        count_ok = False
        notes.append(f"Too many tool calls: {len(result.tool_calls)} > {task.max_tool_calls}")

    nonempty = len(result.response.strip()) > 20
    if not nonempty:
        notes.append("Response too short or empty")

    result.passed = tool_ok and keyword_ok and count_ok and nonempty
    result.score = 1.0 if result.passed else 0.0
    result.notes = "; ".join(notes)
    return result


def llm_judge(prompt: str) -> dict:
    resp = judge_llm.invoke(prompt)
    text = resp.content.strip()
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass
    return {"raw": text}


def score_level2(task: BenchmarkTask, result: TaskResult) -> TaskResult:
    """Workflow scoring: faceted LLM judge on output + method quality."""
    prompt = f"""You are evaluating a data-management agent on a multi-step workflow task.

TASK: {task.description or task.query}
SUCCESS CRITERIA: {task.success_criteria}

AGENT RESPONSE:
{result.response}

TOOLS CALLED (in order): {json.dumps([t['name'] for t in result.tool_calls])}

Score each facet 1-5 (1=poor, 5=excellent). Return ONLY valid JSON:
{{
  "output_correctness": <int>,
  "planning_ordering": <int>,
  "error_recovery": <int>,
  "method_quality": <int>,
  "overall": <int>,
  "rationale": "<brief explanation>"
}}"""
    scores = llm_judge(prompt)
    result.judge_scores = scores
    result.score = scores.get("overall", 0) / 5.0 if isinstance(scores.get("overall"), (int, float)) else None
    result.passed = result.score is not None and result.score >= 0.6
    result.notes = scores.get("rationale", "")
    return result


def score_level3(task: BenchmarkTask, result: TaskResult) -> TaskResult:
    """Discovery scoring: structured hypothesis + ground truth comparison."""
    gt = task.ground_truth_hypothesis or {}
    prompt = f"""You are an expert judge for scientific data-discovery benchmarks (DiscoveryBench-style).

GOAL: {task.description or task.query}
SUCCESS CRITERIA: {task.success_criteria}

EXPERT GROUND TRUTH HYPOTHESIS:
{json.dumps(gt, indent=2)}

AGENT RESPONSE:
{result.response}

Evaluate whether the agent produced a genuine analytical hypothesis with:
- context (domain background)
- variables (measurable quantities)
- relationship (testable claim)
- evidence (data or literature cited)

Score each facet 1-5. Return ONLY valid JSON:
{{
  "context": <int>,
  "variables": <int>,
  "relationship": <int>,
  "evidence_quality": <int>,
  "ground_truth_alignment": <int>,
  "overall": <int>,
  "extracted_hypothesis": {{
    "context": "...",
    "variables": ["..."],
    "relationship": "..."
  }},
  "rationale": "..."
}}"""
    scores = llm_judge(prompt)
    result.judge_scores = scores
    result.score = scores.get("overall", 0) / 5.0 if isinstance(scores.get("overall"), (int, float)) else None
    result.passed = result.score is not None and result.score >= 0.6
    result.notes = scores.get("rationale", "")
    return result


def score_task(task: BenchmarkTask, result: TaskResult) -> TaskResult:
    if task.level == 1:
        return score_level1(task, result)
    if task.level == 2:
        return score_level2(task, result)
    return score_level3(task, result)


def run_benchmark(tasks: List[BenchmarkTask], explorer: DSIExplorer) -> List[TaskResult]:
    results = []
    for task in tasks:
        if task.requires_network and os.environ.get("LEVELS_SKIP_NETWORK", "0") == "1":
            task = BenchmarkTask(**{**asdict(task), "skip": True})
        tr = run_task(explorer, task)
        tr = score_task(task, tr)
        results.append(tr)
        status = "PASS" if tr.passed else ("SKIP" if tr.notes == "skipped" else "FAIL")
        print(f"[{status}] {tr.task_id} — score={tr.score}\n")
    return results


def results_to_dataframe(results: List[TaskResult]) -> pd.DataFrame:
    rows = []
    for r in results:
        row = {
            "task_id": r.task_id,
            "level": r.level,
            "category": r.category,
            "passed": r.passed,
            "score": r.score,
            "elapsed_sec": round(r.elapsed_sec, 1),
            "tokens": r.tokens,
            "tools": ", ".join(t["name"] for t in r.tool_calls),
            "notes": r.notes,
        }
        if r.judge_scores:
            for k, v in r.judge_scores.items():
                if isinstance(v, (int, float, str, bool)):
                    row[f"judge_{k}"] = v
        rows.append(row)
    return pd.DataFrame(rows)


def save_results(results: List[TaskResult], label: str) -> Path:
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    out = RESULTS_DIR / f"{RUN_ID}_{label}.json"
    payload = [asdict(r) for r in results]
    out.write_text(json.dumps(payload, indent=2, default=str))
    print(f"Saved: {out}")
    return out

## Task Catalog

Tasks are drawn from the example queries in `dsi_data_explorer.ipynb`, `dsi_data_explorer-poisson.ipynb`, and `dsi_code_run.ipynb`.

In [16]:
# ---------------------------------------------------------------------------
# Level 1 — Atomic (single tool call, binary pass/fail)
# ---------------------------------------------------------------------------
LEVEL1_TASKS = [
    # --- Search ---
    BenchmarkTask(
        task_id="L1_search_arxiv",
        level=1, category="Search",
        query="can you find some arxiv papers related to inertial confinement fusion",
        description="Single arXiv literature search",
        expected_tools=["arxiv_search_tool"],
        response_keywords=["arxiv", "paper"],
        max_tool_calls=2,
        requires_network=True,
    ),
    # BenchmarkTask(
    #     task_id="L1_search_wikipedia",
    #     level=1, category="Search",
    #     query="can you search wikipedia for this impact: Chelyabinsk, Russia",
    #     description="Single Wikipedia lookup",
    #     expected_tools=["wikipedia_search_tool"],
    #     response_keywords=["chelyabinsk"],
    #     max_tool_calls=2,
    #     requires_network=True,
    # ),
    # BenchmarkTask(
    #     task_id="L1_search_web",
    #     level=1, category="Search",
    #     query="Can you find a websearch on implosion?",
    #     description="Single web search",
    #     expected_tools=["web_search_tool"],
    #     response_keywords=["implosion"],
    #     max_tool_calls=2,
    #     requires_network=True,
    # ),
    # # --- Data Discovery ---
    # BenchmarkTask(
    #     task_id="L1_discovery_list",
    #     level=1, category="Data Discovery",
    #     query="Tell me about the datasets you have.",
    #     description="List datasets from master catalogue",
    #     expected_tools=["query_dsi_tool"],
    #     response_keywords=["dataset"],
    # ),
    # BenchmarkTask(
    #     task_id="L1_discovery_implosion",
    #     level=1, category="Data Discovery",
    #     query="Do you have any implosion data?",
    #     description="Filter catalogue for implosion/NIF-related data",
    #     expected_tools=["query_dsi_tool"],
    #     response_keywords=["nif", "ignition", "implosion", "flash"],
    # ),
    # BenchmarkTask(
    #     task_id="L1_discovery_load_nif",
    #     level=1, category="Data Discovery",
    #     query=f"Can you load the DSI database at {NIF_DB}",
    #     description="Load a specific DSI database by path",
    #     expected_tools=["load_dsi_tool"],
    #     response_keywords=["nif", "load", "table"],
    # ),
    # BenchmarkTask(
    #     task_id="L1_discovery_asteroid",
    #     level=1, category="Data Discovery",
    #     query="I'm interested in asteroid impacts, is there anything related to it?",
    #     description="Search catalogue for asteroid-impact datasets",
    #     expected_tools=["query_dsi_tool"],
    #     response_keywords=["asteroid", "impact", "deep water"],
    # ),
    # # --- Data Preparation ---
    # BenchmarkTask(
    #     task_id="L1_prep_list_tables",
    #     level=1, category="Data Preparation",
    #     query="list the tables/variables",
    #     description="List schema after loading NIF database",
    #     setup_query=f"Load the DSI database at {NIF_DB}",
    #     expected_tools=["query_dsi_tool", "load_dsi_tool"],
    #     response_keywords=["nif", "metadata", "table", "variable"],
    # ),
    # BenchmarkTask(
    #     task_id="L1_prep_sample_rows",
    #     level=1, category="Data Preparation",
    #     query="show me some rows from nif_metadata",
    #     description="Sample rows for validation/inspection",
    #     setup_query=f"Load the DSI database at {NIF_DB}",
    #     expected_tools=["query_dsi_tool"],
    #     response_keywords=["dens", "time", "sim"],
    # ),
    # # --- Metadata Generation ---
    # BenchmarkTask(
    #     task_id="L1_meta_gray_scott",
    #     level=1, category="Metadata Generation",
    #     query="what is this Gray-Scott reaction-diffusion dataset?",
    #     description="Retrieve dataset description from catalogue",
    #     expected_tools=["query_dsi_tool"],
    #     response_keywords=["gray", "scott", "reaction"],
    # ),
    # BenchmarkTask(
    #     task_id="L1_meta_owner",
    #     level=1, category="Metadata Generation",
    #     query="who is the owner of the 3D FLASH National Ignition Facility dataset?",
    #     description="Extract contact/owner metadata",
    #     expected_tools=["query_dsi_tool"],
    #     response_keywords=["sauppe", "joshua", "contact", "owner", "email"],
    # ),
    # # --- Analysis & Summarization ---
    # BenchmarkTask(
    #     task_id="L1_analysis_plot",
    #     level=1, category="Analysis & Summarization",
    #     query="can you create a plot of dens_max over time",
    #     description="Generate a time-series plot from NIF metadata",
    #     setup_query=f"Load the DSI database at {NIF_DB}",
    #     expected_tools=["python_repl_tool", "query_dsi_tool"],
    #     response_keywords=["plot", "dens", "time", "png", "saved"],
    # ),
    # BenchmarkTask(
    #     task_id="L1_analysis_poisson_info",
    #     level=1, category="Analysis & Summarization",
    #     query="Tell me more about the Poisson's Equations for Electrostatic dataset",
    #     description="Summarize a specific dataset entry",
    #     expected_tools=["query_dsi_tool"],
    #     response_keywords=["poisson", "electrostatic"],
    # ),
]

In [ ]:
# ---------------------------------------------------------------------------
# Level 2 — Workflow (multi-step; LLM judge)
# ---------------------------------------------------------------------------
# LEVEL2_TASKS = [
#     BenchmarkTask(
#         task_id="L2_search_summarize",
#         level=2, category="Search",
#         query="Find arxiv papers on asteroid impacts and summarize the most relevant result in 3 sentences.",
#         description="Search literature then synthesize findings",
#         success_criteria="Agent searches arXiv, returns at least one paper, and provides a coherent 3-sentence summary.",
#         requires_network=True,
#     ),
#     BenchmarkTask(
#         task_id="L2_discovery_pipeline",
#         level=2, category="Data Discovery",
#         query="Find implosion-related datasets in the catalogue, load the NIF database, and list all tables with their variables.",
#         description="Discover → load → enumerate schema",
#         success_criteria="Agent queries master DB for NIF/implosion data, loads nif.db, and lists tables/variables.",
#     ),
#     BenchmarkTask(
#         task_id="L2_prep_validate",
#         level=2, category="Data Preparation",
#         query=f"Load {NIF_DB}, query nif_metadata, and report any rows where dens_max is null or zero. Flag them as anomalous.",
#         description="Load → query → validate data quality",
#         success_criteria="Agent loads NIF DB, runs SQL on nif_metadata, and explicitly flags null/zero dens_max values.",
#     ),
#     BenchmarkTask(
#         task_id="L2_meta_record",
#         level=2, category="Metadata Generation",
#         query="Look up the Gray-Scott reaction-diffusion dataset and draft a structured metadata record with title, owner, keywords, and last-updated date.",
#         description="Query catalogue and produce structured metadata",
#         success_criteria="Response includes a structured record with title, contact/owner, keywords, and modification date fields populated.",
#     ),
#     BenchmarkTask(
#         task_id="L2_analysis_pipeline",
#         level=2, category="Analysis & Summarization",
#         query=f"Load {NIF_DB}, query nif_metadata for dens_max and time columns, compute basic statistics (min, max, mean), and create a dens_max vs time plot saved to disk.",
#         description="Query → aggregate → visualize",
#         success_criteria="Agent queries data, reports summary statistics, and saves a plot file.",
#     ),
#     BenchmarkTask(
#         task_id="L2_discovery_poisson",
#         level=2, category="Data Discovery",
#         query=f"Load the poisson database at {POISSON_DB}, list all records, and summarize what data files are available for download.",
#         description="Load poisson DB and enumerate downloadable assets",
#         success_criteria="Agent loads poisson.db, lists records, and identifies HDF5/data URLs.",
#     ),
#     BenchmarkTask(
#         task_id="L2_search_web_wiki",
#         level=2, category="Search",
#         query="do asteroids really impact earth — search the web and wikipedia and give a brief evidence-based answer.",
#         description="Multi-source external search and synthesis",
#         success_criteria="Agent uses web and/or Wikipedia search tools and cites evidence that asteroid impacts occur.",
#         requires_network=True,
#     ),
#     BenchmarkTask(
#         task_id="L2_prep_reload_master",
#         level=2, category="Data Preparation",
#         query=f"Load {NIF_DB}, inspect nif_metadata, then reload the master database and confirm you are back on the catalogue.",
#         description="Context switch: child DB → master DB reload",
#         success_criteria="Agent loads NIF, inspects data, reloads master DB, and confirms catalogue context.",
#     ),
#     BenchmarkTask(
#         task_id="L2_meta_explain_plot",
#         level=2, category="Metadata Generation",
#         query="Create a plot of dens_max over time from the NIF database, then explain step-by-step how you generated it (tools, query, code).",
#         description="Produce artifact + provenance narrative",
#         setup_query=f"Load the DSI database at {NIF_DB}",
#         success_criteria="Agent creates a plot and documents the toolchain: SQL query, Python code, and output path.",
#     ),
#     BenchmarkTask(
#         task_id="L2_analysis_poisson",
#         level=2, category="Analysis & Summarization",
#         query=f"Load {POISSON_DB}, identify the HDF5 data URL for the electrostatic dataset, and summarize the simulation parameters described in the catalogue.",
#         description="Discover poisson assets and summarize parameters",
#         success_criteria="Agent loads poisson DB, finds HDF5 URL, and summarizes grid size / electrostatic context.",
#     ),
# ]

In [ ]:
# ---------------------------------------------------------------------------
# Level 3 — Discovery (hypothesis-level; faceted LLM judge + ground truth)
# ---------------------------------------------------------------------------
# LEVEL3_TASKS = [
#     BenchmarkTask(
#         task_id="L3_search_hypothesis",
#         level=3, category="Search",
#         query="Given interest in asteroid impacts on Earth, search external sources and state a testable hypothesis about impact frequency and energy scale. Structure your answer as context / variables / relationship.",
#         description="Literature-driven hypothesis formation",
#         success_criteria="Agent forms a structured hypothesis with domain context, measurable variables (e.g., diameter, energy, frequency), and a testable relationship.",
#         ground_truth_hypothesis={
#             "context": "Asteroid impacts are documented in geological and observational records; Chelyabinsk-scale events are more frequent than extinction-level impacts.",
#             "variables": ["impactor diameter", "kinetic energy", "impact frequency", "atmospheric entry angle"],
#             "relationship": "Smaller impactors occur more frequently than larger ones; impact energy scales with diameter cubed.",
#         },
#         requires_network=True,
#     ),
#     BenchmarkTask(
#         task_id="L3_discovery_hypothesis",
#         level=3, category="Data Discovery",
#         query="Given a fusion-research goal (understanding implosion dynamics), explore the Ocean 11 catalogue and state which datasets are most relevant and why. Format as context / variables / relationship.",
#         description="Catalogue exploration → relevance hypothesis",
#         success_criteria="Agent identifies NIF/FLASH implosion datasets as most relevant and links them to fusion-implosion research goals.",
#         ground_truth_hypothesis={
#             "context": "Inertial confinement fusion requires understanding implosion symmetry and compression.",
#             "variables": ["dens_max", "simulation time", "geometry (cylindrical)", "NIF shot parameters"],
#             "relationship": "NIF 3D FLASH simulations provide dens_max time series that correlate with implosion compression quality.",
#         },
#     ),
#     BenchmarkTask(
#         task_id="L3_prep_hypothesis",
#         level=3, category="Data Preparation",
#         query=f"Load {NIF_DB}, inspect nif_metadata for data quality issues, and hypothesize what physical or pipeline factors could explain any anomalies in dens_max. Structure as context / variables / relationship.",
#         description="Data quality inspection → causal hypothesis",
#         success_criteria="Agent inspects data, identifies potential anomalies, and proposes a physically plausible explanation.",
#         ground_truth_hypothesis={
#             "context": "NIF simulation metadata tracks peak density over time during implosion.",
#             "variables": ["dens_max", "time", "sim_name", "timestep"],
#             "relationship": "Outliers or drops in dens_max may indicate simulation restart artifacts, mesh refinement events, or failed convergence at specific timesteps.",
#         },
#     ),
#     BenchmarkTask(
#         task_id="L3_meta_hypothesis",
#         level=3, category="Metadata Generation",
#         query="Examine the Gray-Scott reaction-diffusion dataset entry and hypothesize what additional metadata fields would improve reproducibility. Structure as context / variables / relationship.",
#         description="Metadata gap analysis → improvement hypothesis",
#         success_criteria="Agent reviews existing metadata and proposes missing fields (e.g., PDE parameters, boundary conditions, solver version).",
#         ground_truth_hypothesis={
#             "context": "Gray-Scott is a reaction-diffusion PDE system requiring initial conditions and feed/kill parameters.",
#             "variables": ["diffusion coefficients (Du, Dv)", "feed rate (f)", "kill rate (k)", "grid resolution", "time step", "solver"],
#             "relationship": "Missing PDE parameters and solver configuration metadata reduce reproducibility of pattern-formation results.",
#         },
#     ),
#     BenchmarkTask(
#         task_id="L3_analysis_hypothesis",
#         level=3, category="Analysis & Summarization",
#         query=f"Load {NIF_DB}, analyze the relationship between simulation time and dens_max in nif_metadata, and state a structured hypothesis about implosion compression dynamics. Format: context / variables / relationship.",
#         description="Data analysis → scientific hypothesis",
#         success_criteria="Agent queries dens_max vs time, observes trend, and states a hypothesis about compression behavior.",
#         ground_truth_hypothesis={
#             "context": "NIF cylindrical implosion simulations track peak density evolution.",
#             "variables": ["time", "dens_max", "sim_name"],
#             "relationship": "dens_max generally increases during early implosion phases as the target compresses, with plateaus or drops at late times indicating shock reflection or disassembly.",
#         },
#     ),
#     BenchmarkTask(
#         task_id="L3_discovery_poisson_hypothesis",
#         level=3, category="Data Discovery",
#         query=f"Explore {POISSON_DB} and the electrostatic Poisson dataset, then hypothesize how grid resolution affects field accuracy. Structure as context / variables / relationship.",
#         description="Dataset exploration → resolution-accuracy hypothesis",
#         success_criteria="Agent identifies 32x32 grid electrostatic simulations and hypothesizes resolution-accuracy tradeoff.",
#         ground_truth_hypothesis={
#             "context": "Poisson's equation governs electrostatic potential; numerical solutions depend on grid resolution.",
#             "variables": ["grid size (32x32)", "potential field error", "charge distribution", "boundary conditions"],
#             "relationship": "Finer grid resolution reduces discretization error in the electrostatic potential field.",
#         },
#     ),
# ]

# ALL_TASKS = LEVEL1_TASKS + LEVEL2_TASKS + LEVEL3_TASKS
# print(f"Task catalog: {len(LEVEL1_TASKS)} L1 + {len(LEVEL2_TASKS)} L2 + {len(LEVEL3_TASKS)} L3 = {len(ALL_TASKS)} total")

## Run Benchmarks

Set `RUN_MODE = "smoke"` to run one task per category per level (fast sanity check).  
Set `RUN_MODE = "full"` to run the complete catalog.

Set environment variable `LEVELS_SKIP_NETWORK=1` to skip tasks that require external network access.

In [17]:
def select_tasks(tasks: List[BenchmarkTask], mode: str) -> List[BenchmarkTask]:
    if mode == "full":
        return tasks
    # smoke: first task per (level, category) pair
    seen = set()
    selected = []
    for t in tasks:
        key = (t.level, t.category)
        if key not in seen:
            seen.add(key)
            selected.append(t)
    return selected


def run_level(label: str, tasks: List[BenchmarkTask]) -> List[TaskResult]:
    selected = select_tasks(tasks, RUN_MODE)
    display(Markdown(f"## Running {label} ({len(selected)} tasks, mode={RUN_MODE})"))
    explorer = make_explorer()
    results = run_benchmark(selected, explorer)
    save_results(results, label)
    return results

### Level 1 — Atomic

In [18]:
l1_results = run_level("level1_atomic", LEVEL1_TASKS)
l1_df = results_to_dataframe(l1_results)
display(l1_df)

## Running level1_atomic (1 tasks, mode=smoke)

Creating workspace at: /Users/hhn/lanl/dsi/tools/ai_search/dsi_explorer_agent__2026_06_30__11_26


### `L1_search_arxiv` — Level 1 / Search

**Query:** can you find some arxiv papers related to inertial confinement fusion

Here are some arXiv papers related to inertial confinement fusion:

- **Fuel Target Implosion in Ion beam Inertial Confinement Fusion**  
  Shigeo Kawata, 2015  
  https://arxiv.org/abs/1504.01831v1  
  Focuses on target implosion physics in ion-beam-driven inertial fusion.

- **How much laser power can propagate through fusion plasma?**  
  Pavel M. Lushnikov, Harvey A. Rose, 2005  
  https://arxiv.org/abs/physics/0512271v3  
  Relevant to laser propagation and self-focusing issues in ICF plasma.

- **Magnetized Plasma Target for Plasma-Jet-Driven Magneto-Inertial Fusion**  
  Scott C. Hsu, Samuel J. Langendorf, 2018  
  https://arxiv.org/abs/1803.03323v2  
  More on magneto-inertial fusion, but closely related to inertial fusion concepts.

- **Retrospective of the ARPA-E ALPHA fusion program**  
  C. L. Nehl et al., 2019  
  https://arxiv.org/abs/1907.09921v2  
  Broad overview of pulsed fusion approaches, including magneto-inertial fusion.

A couple of the search results were unrelated because “inertial” and “fusion” matched in other contexts.

If you want, I can do a more targeted search for:
- **laser-driven ICF**
- **NIF ignition**
- **ICF instabilities / hohlraums / implosion symmetry**
- **recent review papers**

Tools called: ['arxiv_search_tool'] | 13.6s | 4290 tokens
[PASS] L1_search_arxiv — score=1.0

Saved: /Users/hhn/lanl/dsi/tools/ai_search/levels_benchmark_results/20260630_112522_level1_atomic.json


,task_id,level,category,passed,score,elapsed_sec,tokens,tools,notes
0,L1_search_arxiv,1,Search,True,1.0,13.6,4290,arxiv_search_tool,


#### Atomic failure analysis

When Level 1 fails, the issue is usually **tool selection** (wrong or missing tool) or **parameter precision** (wrong DB path, table name, or query). Review `notes` and `tools` columns to see whether failures are comprehension vs. execution.

In [19]:
if not l1_df.empty:
    fails = l1_df[l1_df["passed"] == False]
    if len(fails):
        display(Markdown("**Failed atomic tasks:**"))
        display(fails[["task_id", "category", "tools", "notes"]])
    else:
        print("All atomic tasks passed.")
    print(f"\nAtomic pass rate: {l1_df['passed'].mean():.0%} ({l1_df['passed'].sum()}/{len(l1_df)})")

All atomic tasks passed.

Atomic pass rate: 100% (1/1)


### Level 2 — Workflow

In [ ]:
# l2_results = run_level("level2_workflow", LEVEL2_TASKS)
# l2_df = results_to_dataframe(l2_results)
# display(l2_df)

### Level 3 — Discovery

In [ ]:
# l3_results = run_level("level3_discovery", LEVEL3_TASKS)
# l3_df = results_to_dataframe(l3_results)
# display(l3_df)

## Aggregate Results

In [ ]:
all_results = l1_results 
# + l2_results + l3_results
summary_df = results_to_dataframe(all_results)

display(Markdown("### Pass rate by level and category"))
pivot = summary_df.pivot_table(
    index="category", columns="level", values="passed", aggfunc="mean"
).round(2)
display(pivot)

display(Markdown("### Mean score by level and category"))
score_pivot = summary_df.pivot_table(
    index="category", columns="level", values="score", aggfunc="mean"
).round(2)
display(score_pivot)

display(Markdown("### Full results"))
display(summary_df)

save_results(all_results, "all_levels_summary")

## Interactive Single-Task Runner

Use this cell to debug individual tasks or add custom queries.

In [ ]:
def debug_task(task_id: str):
    task = next(t for t in ALL_TASKS if t.task_id == task_id)
    explorer = make_explorer()
    tr = run_task(explorer, task)
    tr = score_task(task, tr)
    print(f"Passed: {tr.passed} | Score: {tr.score}")
    if tr.judge_scores:
        print(json.dumps(tr.judge_scores, indent=2))
    return tr

# Example: debug_task("L1_discovery_implosion")

## Optional: MCP Mode

To benchmark via MCP (same tools exposed by `mcp_wrapper_server.py`):

1. In a terminal: `python mcp_wrapper_server.py streamable-http`
2. Set `USE_MCP = True` in the config cell and re-run setup.

This tests the same tool surface through the MCP transport layer used in production integrations.